In [ ]:
from types import SimpleNamespace
import json
from pathlib import Path
import warnings
from tensorflow.keras.callbacks import ModelCheckpoint

from utils import create_new_experiment
from callbacks import WarmupSchedule, SamplesSeen
from constants import DATA_ROOT, RESULTS_ROOT
from dataset import Dataset
from model import RNABagModel

In [ ]:
# tf.keras.mixed_precision.set_global_policy('mixed_float16')

In [ ]:
warnings.filterwarnings(
    "ignore",
    message="Observation names are not unique"
)

In [ ]:
# CONFIGURE RUN

CONFIG = {
    'project': 'scUNVEIL Training',
    'run_name': '',

    'emb_dim': 2048,
    'n_layers': 7,

    'n_genes': 60_000,
    'backed_r': True,

    'lr': 1e-4,

    'warmup_steps': 10_000,
    'total_steps': 1_000_000,

    'batch_size': 1000,

    'steps_per_epoch': 10_000,
    'limit_loaded_shards': None,
}
CF = SimpleNamespace(**CONFIG)

In [ ]:
# TRACKING & CHECKPOINT DIR INIT

# Create Result Directory
experiment_id = create_new_experiment(CONFIG)

# Initialize W&B only if login file exists
wandb_run = None
wandb_login_path = Path("wandb_login.json")
use_wandb = wandb_login_path.exists()

if use_wandb:
    import wandb
    
    with open(wandb_login_path, "r", encoding="utf-8") as fr:
        wandb_login = json.load(fr)

    wandb.login(key=wandb_login["key"])

    wandb_run = wandb.init(
        project=CF.project,
        entity=wandb_login["entity"],
        name=f"emb={CF.emb_dim} l={CF.n_layers} g={int(round(CF.n_genes / 1000))}k lr={CF.lr:.0e} bs={CF.batch_size}",
        config=CF,
        id=experiment_id
    )
else:
    print(
    "wandb_login.json not found. Skipping W&B initialization. "
    "If you want to use W&B tracking, create a wandb_login.json file with the format: "
    '{"key": "YOUR_WANDB_API_KEY", "entity": "YOUR_WANDB_ENTITY"}'
)

In [ ]:
# DATASET INIT

data = Dataset(
    source_dir=DATA_ROOT / 'shuffled_shards', 
    backed_r=CF.backed_r, 
    limit_loaded_shards=CF.limit_loaded_shards,
)

dg_train = data.pretrain_dg(CF.batch_size, ds='train', max_genes=CF.n_genes)

In [ ]:
# MODEL INIT

m = RNABagModel(n_vars=CF.n_genes, n_layers=CF.n_layers, emb_dim=CF.emb_dim)

lr_schedule = WarmupSchedule(
    base_lr=CF.lr, 
    warmup_steps=CF.warmup_steps, 
    finish_lr=0.0, 
    decay_steps=CF.total_steps - CF.warmup_steps
)

m.compile_pretrain(lr_schedule)

In [ ]:
# TRAINING LOOP

train_callbacks = [
    SamplesSeen(batch_size=CF.batch_size),
    ModelCheckpoint(
        str(RESULTS_ROOT / 'training' / experiment_id / 'weights' / '{epoch:04d}.weights.h5'),
        save_best_only=False,
        save_weights_only=True,
    )
]

if use_wandb:
    from wandb.integration.keras import WandbMetricsLogger
    train_callbacks.append(WandbMetricsLogger())

m.model.fit(
    dg_train,
    epochs=CF.total_steps // CF.steps_per_epoch,
    callbacks=train_callbacks,
    steps_per_epoch=CF.steps_per_epoch
)